[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/data-engineer-accelerator/blob/main/notebooks/M01_DE_Foundations.ipynb)

# Module 01: Data Engineering Foundations

**Program:** Quintrix Jr. Data Engineer Training  
**Duration:** 3 hours  
**Instructor:** Sunil Mogadati

---

**Today's Agenda:**

| # | Topic | Time |
|---|-------|------|
| 1 | Data Engineering Role and Responsibilities | 35 min |
| 2 | Data Lifecycle: Generation → Storage → Processing → Serving | 40 min |
| 3 | ETL vs. ELT Paradigms | 40 min |
| 4 | Batch vs. Real-Time Processing | 30 min |
| 5 | Modern Data Stack Overview | 35 min |

---

## 1. Data Engineering Role and Responsibilities

**One-line answer:** Data engineers build the systems that move data from where it's generated to where it's useful.

You don't analyze data. You don't build dashboards. You don't train models. You build the **infrastructure** that makes all of that possible.

> **Analogy:** If data scientists are chefs, data engineers build the kitchen — the plumbing, the gas lines, the refrigeration, the delivery trucks. No kitchen, no cooking.

### What Data Engineers Actually Do

| Activity | Example |
|----------|----------|
| **Build data pipelines** | Write PySpark jobs that transform raw call logs into clean, queryable tables |
| **Design data models** | Create star schemas so analysts can answer business questions with simple JOINs |
| **Move data between systems** | Extract from MongoDB, transform with Spark, load into BigQuery |
| **Ensure data quality** | Catch duplicates, validate schemas, mask PII before data reaches reports |
| **Orchestrate workflows** | Schedule Airflow DAGs that run pipelines daily at 6 AM |
| **Optimize performance** | Fix a Spark job that takes 4 hours to run in 20 minutes |
| **Monitor and troubleshoot** | Get an alert at 3 AM that yesterday's data didn't load, diagnose and fix |
| **Maintain infrastructure** | Manage cloud resources, databases, clusters, storage |

### How Data Engineers Differ from Other Roles

| | Data Engineer | Data Analyst | Data Scientist | ML Engineer |
|---|---|---|---|---|
| **Primary question** | "How do we get the data there?" | "What does the data say?" | "Can we predict/model this?" | "How do we deploy this model?" |
| **Main tools** | Spark, Airflow, SQL, Cloud | SQL, Tableau, Excel | Python, R, Jupyter | PyTorch, MLflow, Docker |
| **Output** | Pipelines, tables, APIs | Reports, dashboards | Models, insights | Production model services |
| **Closest analogy** | Plumber / Electrician | Accountant | Research scientist | Factory engineer |
| **Thinks about** | Throughput, latency, reliability | Trends, KPIs, anomalies | Accuracy, features, bias | Inference speed, scaling |

> **Key insight:** In many companies, especially startups and mid-size firms, these roles blur. You'll meet "Analytics Engineers" (DE + DA) and "ML Engineers" who also build their own pipelines. Knowing DE makes you more valuable in ANY data role.

### Day in the Life of a Jr. Data Engineer

**9:00 AM** — Check Airflow dashboard. Last night's daily pipeline ran successfully. One warning: a source file had 3% more null values than usual.

**9:30 AM** — Standup. PM asks: "Can we add a new column to the daily report — average call duration by campaign?" You say: "Yes, I can add it to the Silver layer transform and the BigQuery mart. Probably 2 hours."

**10:00 AM** — Write a PySpark transformation to compute avg call duration grouped by campaign. Add it to the existing pipeline.

**11:30 AM** — Write a data quality test: assert avg_duration is between 30 and 1800 seconds. If it's outside that range, something is wrong with the source data.

**1:00 PM** — Code review from senior DE. They suggest using a window function instead of a group-by + join. You learn something.

**2:00 PM** — Deploy the updated pipeline via CI/CD (GitHub Actions). It runs in the staging environment first.

**3:00 PM** — Investigate a data quality alert from yesterday. An upstream system changed their date format from `MM/DD/YYYY` to `YYYY-MM-DD`. Your pipeline broke. You add a format-detection step.

**4:30 PM** — Document the schema change in the data dictionary. Update the runbook.

> **Notice:** Most of the day is writing Python/SQL, debugging data issues, and working with other engineers. It's not abstract — it's very hands-on.

---

## 2. Data Lifecycle: Generation → Storage → Processing → Serving

Every piece of data in every company follows the same journey:

```
GENERATION  →  STORAGE  →  PROCESSING  →  SERVING
(something     (put it      (clean it,     (make it
 happens)      somewhere)    transform)     useful)
```

| Stage | What Happens | Example (Call Center) |
|-------|-------------|----------------------|
| **Generation** | An event occurs that produces data | A customer calls and the AI agent records the call |
| **Storage** | Raw data lands somewhere | Call record is saved as JSON in MongoDB |
| **Processing** | Data is cleaned, validated, transformed, enriched | Deduplicate calls, convert UTC→EST, join call to order |
| **Serving** | Processed data is made available for consumption | Daily campaign report shows conversion rates in BigQuery |

> **Your job as a DE** is stages 2, 3, and 4. You don't control what generates the data (that's the application team), but everything after that is yours.

In [ ]:
# The Data Lifecycle in Python — a simple mental model

# Stage 1: GENERATION — something happens in the real world
raw_event = {
    "call_id": "call_ac567d48edabbd24be4eaad",
    "dnis": "8005551234",
    "caller_phone": "3135559876",
    "start_time": "2026-03-10T19:30:00Z",   # UTC!
    "end_time": "2026-03-10T19:37:45Z",
    "disposition": "completed",
    "customer": {
        "first_name": "Jane",
        "last_name": "Smith",
        "city": "Detroit",
        "state": "MI"
    }
}
print("Stage 1 — GENERATION: Raw event from the VA platform")
print(f"  Call ID: {raw_event['call_id']}")
print(f"  Raw timestamp: {raw_event['start_time']} (UTC)")
print(f"  Customer nested inside: {raw_event['customer']}")
print()

In [ ]:
import json

# Stage 2: STORAGE — save the raw data as-is (Bronze layer)
# In production: write to cloud storage (GCS, S3) or append to a Delta Lake table
# Here: simulate by writing to a JSON file

raw_json = json.dumps(raw_event, indent=2)
print("Stage 2 — STORAGE: Raw JSON saved exactly as received")
print(raw_json)
print()
print("Rule: NEVER modify raw data. Store it as-is. You might need it later.")

In [ ]:
from datetime import datetime, timezone, timedelta

# Stage 3: PROCESSING — clean, transform, enrich (Silver layer)
# This is where most of your work as a DE happens

def process_call(raw):
    """Transform raw call event into clean, analysis-ready record."""
    
    # Parse timestamps
    start_utc = datetime.fromisoformat(raw["start_time"].replace("Z", "+00:00"))
    end_utc = datetime.fromisoformat(raw["end_time"].replace("Z", "+00:00"))
    
    # Convert UTC → EST (real bug: reports showed wrong date for evening calls)
    est_offset = timedelta(hours=-5)
    start_est = start_utc + est_offset
    
    # Flatten nested customer object
    customer = raw.get("customer", {})
    
    # Calculate duration
    duration = (end_utc - start_utc).total_seconds()
    
    return {
        "call_id": raw["call_id"],
        "dnis": raw["dnis"],
        "caller_phone": raw["caller_phone"],
        "start_time_utc": start_utc.isoformat(),
        "start_time_est": start_est.isoformat(),
        "call_date": start_est.strftime("%Y-%m-%d"),  # EST date for reporting
        "call_hour": start_est.hour,
        "duration_seconds": duration,
        "disposition": raw["disposition"],
        # Flattened customer fields
        "customer_first_name": customer.get("first_name"),
        "customer_last_name": customer.get("last_name"),
        "customer_city": customer.get("city"),
        "customer_state": customer.get("state"),
    }

clean_record = process_call(raw_event)

print("Stage 3 — PROCESSING: Cleaned and transformed record")
print(f"  UTC start:  {raw_event['start_time']}")
print(f"  EST start:  {clean_record['start_time_est']}")
print(f"  Call date:  {clean_record['call_date']} (EST — this is the date reports use)")
print(f"  Call hour:  {clean_record['call_hour']} (2:30 PM EST — not 7:30 PM UTC!)")
print(f"  Duration:   {clean_record['duration_seconds']}s ({clean_record['duration_seconds']/60:.1f} min)")
print(f"  Customer:   {clean_record['customer_first_name']} {clean_record['customer_last_name']}, {clean_record['customer_city']}, {clean_record['customer_state']}")
print()
print("Notice: nested JSON is flattened, UTC is converted, duration is calculated.")
print("This is what a DE does — make raw data usable.")

In [ ]:
# Stage 4: SERVING — make it available for reports/analysis (Gold layer)
# In production: write to BigQuery, create materialized views, power dashboards
# Here: simulate with a simple aggregation

# Simulate a batch of processed calls
daily_calls = [
    {"campaign": "Campaign A", "disposition": "completed", "duration": 465, "amount": 79.99},
    {"campaign": "Campaign A", "disposition": "completed", "duration": 380, "amount": 119.98},
    {"campaign": "Campaign A", "disposition": "dropped",   "duration": 45,  "amount": 0},
    {"campaign": "Campaign A", "disposition": "completed", "duration": 510, "amount": 79.99},
    {"campaign": "Campaign A", "disposition": "dropped",   "duration": 30,  "amount": 0},
    {"campaign": "Campaign B", "disposition": "completed", "duration": 390, "amount": 49.95},
    {"campaign": "Campaign B", "disposition": "completed", "duration": 420, "amount": 49.95},
    {"campaign": "Campaign B", "disposition": "transferred", "duration": 200, "amount": 0},
    {"campaign": "Campaign B", "disposition": "dropped",   "duration": 15,  "amount": 0},
    {"campaign": "Campaign B", "disposition": "completed", "duration": 350, "amount": 49.95},
]

# Build a simple report — this is what the Gold layer produces
from collections import defaultdict

stats = defaultdict(lambda: {"total": 0, "completed": 0, "revenue": 0.0, "total_duration": 0})

for call in daily_calls:
    c = call["campaign"]
    stats[c]["total"] += 1
    stats[c]["total_duration"] += call["duration"]
    if call["disposition"] == "completed":
        stats[c]["completed"] += 1
        stats[c]["revenue"] += call["amount"]

print("Stage 4 — SERVING: Daily Campaign Report (Gold Layer)")
print("=" * 75)
print(f"{'Campaign':<18} {'Calls':>6} {'Converted':>10} {'Conv %':>8} {'Revenue':>10} {'Avg Dur':>8}")
print("-" * 75)
for campaign, s in stats.items():
    conv_rate = (s["completed"] / s["total"] * 100) if s["total"] > 0 else 0
    avg_dur = s["total_duration"] / s["total"] if s["total"] > 0 else 0
    print(f"{campaign:<18} {s['total']:>6} {s['completed']:>10} {conv_rate:>7.1f}% {s['revenue']:>9.2f} {avg_dur:>7.0f}s")
print("=" * 75)
print()
print("This report is what business stakeholders see.")
print("Your job: make sure the data is correct, timely, and reliable.")

---

## 3. ETL vs. ELT Paradigms

Two fundamental patterns for moving data. You'll use both in your career.

### ETL: Extract → Transform → Load

```
Source  ──→  Transform (outside the warehouse)  ──→  Load into warehouse
```

- Transform happens **before** the data reaches the warehouse
- The warehouse only sees clean data
- Traditional approach (Informatica, SSIS, custom scripts)
- You need compute power **outside** the warehouse (Spark cluster, Python server)

### ELT: Extract → Load → Transform

```
Source  ──→  Load raw into warehouse  ──→  Transform inside the warehouse
```

- Raw data lands in the warehouse first
- Transform happens **inside** the warehouse using SQL
- Modern approach (BigQuery, Snowflake, Databricks)
- The warehouse has enough compute to handle transformations

### Comparison

| Aspect | ETL | ELT |
|--------|-----|-----|
| **Transform happens** | Before loading (external compute) | After loading (warehouse compute) |
| **Raw data in warehouse?** | No — only clean data arrives | Yes — raw data lands first |
| **Compute** | External (Spark, Python, etc.) | Warehouse engine (BigQuery, Snowflake) |
| **Best for** | Complex transforms, legacy systems, multiple targets | Cloud warehouses with cheap compute |
| **Audit trail** | Harder — raw data may not be preserved | Easier — raw data is always in the warehouse |
| **Cost model** | Pay for external compute cluster | Pay for warehouse processing |
| **When to use** | PySpark transforms, ML pipelines, multi-system | SQL-heavy transforms, analyst self-service |

In [ ]:
# ETL Example — Transform BEFORE loading

# Raw data from source system
raw_calls = [
    {"id": "call_001", "ts": "2026-03-10T19:30:00Z", "dur": "7m45s", "status": "COMPLETED"},
    {"id": "call_002", "ts": "2026-03-10T20:15:00Z", "dur": "3m10s", "status": "dropped"},
    {"id": "call_001", "ts": "2026-03-10T19:30:00Z", "dur": "7m45s", "status": "COMPLETED"},  # duplicate!
    {"id": "call_003", "ts": "2026-03-10T21:00:00Z", "dur": None,    "status": "COMPLETED"},  # null duration!
]

print("=== ETL: Transform FIRST, then load ===")
print(f"\nRaw records: {len(raw_calls)}")

# --- TRANSFORM (happens outside the warehouse) ---

def parse_duration(dur_str):
    """Convert '7m45s' to seconds."""
    if dur_str is None:
        return 0  # default for nulls
    minutes = int(dur_str.split('m')[0])
    seconds = int(dur_str.split('m')[1].replace('s', ''))
    return minutes * 60 + seconds

# Step 1: Deduplicate
seen_ids = set()
deduped = []
for call in raw_calls:
    if call["id"] not in seen_ids:
        seen_ids.add(call["id"])
        deduped.append(call)

# Step 2: Clean and standardize
clean_calls = []
for call in deduped:
    clean_calls.append({
        "call_id": call["id"],
        "start_time": call["ts"],
        "duration_seconds": parse_duration(call["dur"]),
        "disposition": call["status"].lower(),  # normalize case
    })

print(f"After dedup: {len(clean_calls)} records")
print(f"\nClean data ready to LOAD into warehouse:")
for c in clean_calls:
    print(f"  {c}")

print("\n→ The warehouse only sees clean, deduplicated, standardized data.")
print("→ But: if the transform had a bug, the raw data is gone. No way to re-derive.")

In [ ]:
# ELT Example — Load raw FIRST, transform inside the warehouse

print("=== ELT: Load raw FIRST, transform in warehouse ===")
print(f"\nRaw records: {len(raw_calls)}")

# --- LOAD (dump everything into the warehouse as-is) ---
import sqlite3

conn = sqlite3.connect(":memory:")  # simulates BigQuery/Snowflake
cursor = conn.cursor()

# Bronze table — raw data, no cleaning
cursor.execute("""
    CREATE TABLE bronze_calls (
        id TEXT,
        ts TEXT,
        dur TEXT,
        status TEXT
    )
""")

for call in raw_calls:
    cursor.execute("INSERT INTO bronze_calls VALUES (?, ?, ?, ?)",
                   (call["id"], call["ts"], call["dur"], call["status"]))

print("Step 1 — LOADED raw data into bronze_calls (duplicates, nulls, and all):")
for row in cursor.execute("SELECT * FROM bronze_calls"):
    print(f"  {row}")

# --- TRANSFORM (happens inside the warehouse using SQL) ---
cursor.execute("""
    CREATE TABLE silver_calls AS
    SELECT DISTINCT
        id AS call_id,
        ts AS start_time,
        COALESCE(dur, '0m0s') AS duration_raw,
        LOWER(status) AS disposition
    FROM bronze_calls
""")

print("\nStep 2 — TRANSFORMED inside warehouse (silver_calls):")
for row in cursor.execute("SELECT * FROM silver_calls"):
    print(f"  {row}")

print("\n→ Raw data is PRESERVED in bronze_calls. If transform has a bug, re-run it.")
print("→ This is why ELT is preferred in modern data engineering.")

conn.close()

### In Practice: Most Pipelines Blend Both

Most modern data pipelines use a hybrid approach:

- **Light ETL** on the way in: basic schema enforcement, format conversion, deduplication (PySpark)
- **Heavy ELT** in the warehouse: aggregations, window functions, business logic (BigQuery SQL)

This combination gives you the best of both worlds — complex transforms where you need code (PySpark), simple transforms where SQL is faster (BigQuery).

> **In this program:** You'll use PySpark for complex transforms (ETL-style) and BigQuery for analytical queries (ELT-style). You'll see both patterns in action starting in Module 06.

---

## 4. Batch vs. Real-Time Processing

| Aspect | Batch Processing | Real-Time (Stream) Processing |
|--------|-----------------|------------------------------|
| **When it runs** | On a schedule (hourly, daily, weekly) | Continuously, as events arrive |
| **Data volume** | Large chunks at once | One event at a time |
| **Latency** | Minutes to hours | Milliseconds to seconds |
| **Tools** | Spark, Airflow, dbt | Kafka, Flink, Spark Streaming |
| **Complexity** | Lower — well-understood patterns | Higher — ordering, exactly-once, state |
| **Cost** | Lower — compute only when running | Higher — compute always running |
| **Example** | "Generate yesterday's report at 6 AM" | "Alert me the moment a payment fails" |

### Where Each Fits

| Use Case | Type | Why |
|----------|------|-----|
| Daily campaign performance report | **Batch** | Stakeholders check it every morning — doesn't need to be real-time |
| Refresh reporting tables | **Batch** | Can run on a schedule (every 15 min, hourly, daily) |
| Payment failure alert | **Real-time** | Need to know immediately if the payment gateway is down |
| Fraud detection | **Real-time** | Can't wait for a batch to catch a stolen card |
| Monthly revenue reconciliation | **Batch** | Runs once a month, processes millions of records |
| Live agent availability dashboard | **Real-time** | Supervisors need live view of who's on calls |

> **In this program:** We focus on **batch processing**. That's where 90% of DE work happens, and it's where Jr. DEs start. Real-time (Kafka, Flink) is a specialization you'll add later in your career.

### Micro-Batch: The Middle Ground

Many production systems run batch on a tight schedule (every 5-15 minutes). This is called **micro-batch** — not truly real-time, but frequent enough for most reporting needs. It's simpler to build and operate than a full streaming pipeline.

```
True Batch:    |──── 24 hrs ────|──── 24 hrs ────|  (daily runs)
Micro-Batch:   |--15m--|--15m--|--15m--|--15m--|  (frequent runs)
Real-Time:     ───────────────────────────  (continuous)
```

---

## 5. Modern Data Stack Overview

The tools and infrastructure that modern data teams use. This has changed dramatically in the last 5 years.

### Why the Shift?

Three forces drove the change from legacy to modern data stacks:

1. **Cloud economics:** Separation of compute and storage. Pay for what you use instead of buying servers.
2. **Data volume:** Terabytes to petabytes — traditional tools (SSIS, stored procedures) couldn't keep up.
3. **Code-based pipelines:** Version-controlled, testable, reviewable — data engineering became software engineering.

### Then vs. Now

| Layer | Legacy (2015) | Modern (2025+) |
|-------|--------------|----------------|
| **Ingestion** | Informatica, SSIS, custom scripts | Fivetran, Airbyte, custom PySpark |
| **Storage** | On-prem SQL Server, Oracle | Cloud: BigQuery, Snowflake, Databricks, S3/GCS |
| **Processing** | Stored procedures, SSIS packages | PySpark, dbt, SQL in warehouse |
| **Table Format** | Regular tables | Delta Lake, Apache Iceberg (ACID on data lakes) |
| **Orchestration** | Windows Task Scheduler, cron, SSIS | Apache Airflow, Dagster, Prefect |
| **CI/CD** | Manual deployment | GitHub Actions, automated testing |
| **Quality** | Manual checks, hope | Great Expectations, dbt tests, automated validation |
| **Monitoring** | "Did anyone complain?" | Alerts, dashboards, SLAs |

### The Medallion Architecture

A key pattern in the modern data stack is **layered storage** — often called the medallion architecture:

```
BRONZE          →  SILVER            →  GOLD
(raw, as-is)       (cleaned, joined)     (business-ready, aggregated)
```

| Layer | Purpose | Key Property |
|-------|---------|-------------|
| **Bronze** | Store raw data exactly as received | Append-only, immutable, full audit trail |
| **Silver** | Clean, deduplicate, validate, join | Quality-checked, conformed schemas |
| **Gold** | Business-ready marts for reporting | Pre-aggregated, optimized for queries |

Why 3 layers? **Single responsibility.** Bronze never changes (audit trail). Silver is where bugs get caught. Gold is what business users see. If Gold has bad data, fix Silver and re-derive.

### The Stack You'll Learn in This Program

| Layer | Tool | Module |
|-------|------|--------|
| **Language** | Python, SQL | M03, M06 |
| **Processing** | PySpark (Apache Spark) | M06, M07 |
| **Cloud Platform** | GCP (BigQuery, Dataproc, Cloud Storage) | M08 |
| **Table Format** | Delta Lake, Apache Iceberg | M09, M10 |
| **Orchestration** | Apache Airflow | M11 |
| **CI/CD** | GitHub Actions | M12 |
| **Version Control** | Git | M02 |
| **Data Modeling** | Star schema, dimensional modeling | M04 |
| **Legacy Context** | Hadoop, Hive | M05 |

> **Why GCP?** BigQuery is the fastest-growing cloud data warehouse. Dataproc gives you managed Spark. Cloud Composer gives you managed Airflow. One ecosystem, one billing console. The concepts transfer to AWS and Azure.

---

## Key Takeaways

1. **Data engineers build the infrastructure** that moves data from where it's generated to where it's useful — pipelines, storage, processing, serving.
2. **The data lifecycle** is generation → storage → processing → serving. You own stages 2-4.
3. **ETL transforms before loading** (external compute); **ELT loads first, transforms in the warehouse** (warehouse compute). Modern pipelines blend both.
4. **Batch processing** handles the vast majority of DE work — scheduled runs that process data in chunks. Real-time is a specialization you'll add later.
5. **The modern data stack** has shifted from on-prem tools (SSIS, stored procedures, Task Scheduler) to cloud-native tools (PySpark, BigQuery, Airflow, GitHub Actions).
6. **The medallion architecture** (Bronze → Silver → Gold) separates raw storage from clean data from business-ready marts.

---

## Discussion Questions

1. **Where have you encountered data quality issues** in your previous work or studies? What happened?
2. **Can you think of a case** where batch processing wouldn't be fast enough? What would you need instead?
3. **Why has the data stack shifted from on-prem to cloud?** What are the advantages? Any disadvantages?
4. **When would you choose ETL over ELT?** Think about a scenario where each is the better fit.

---

## Homework (Before Next Session)

1. **Verify your setup from M00** — run the verification checklist:
   ```bash
   python3 --version          # Python 3.10+
   pip3 --version             # pip (package manager)
   git --version              # Git 2.x+
   code --version             # VS Code
   ```
   If anything is missing, refer to the Setup & Installation section in the M00 notebook.
2. **Configure Git identity** (if you haven't):
   ```bash
   git config --global user.name "Your Name"
   git config --global user.email "your.email@example.com"
   ```
3. **Review SQL basics** if rusty:
   - [SQLBolt](https://sqlbolt.com/) — Lessons 1-6 (30 min)
   - Know: SELECT, WHERE, JOIN, GROUP BY, ORDER BY
4. **Reflect on today's discussion questions** — bring one real-world example of a data quality issue you've seen or read about.

---

**Next session:** M02 (Git & Linux) + start M03 (Advanced SQL)